# 03 — Model Training

**Goal:** Train the XGBoost default classifier, calibrate Traffic Light thresholds,
and validate performance.

## Contents
1. Load master features
2. Time-aware train/val/test split (no data leakage)
3. Baseline: Logistic Regression scorecard
4. Core model: XGBoost with calibrated probabilities
5. Cost-benefit threshold optimisation → Traffic Light bands
6. Evaluation: ROC-AUC, Gini, KS, PR-AUC
7. Traffic Light band default rates (validation)
8. Save model artefacts

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.preprocessing import StandardScaler

from models import DefaultClassifier
from training.evaluate import evaluate_model, _ks_statistic
from training.threshold_optimizer import optimise_thresholds, threshold_sweep_report
from utils.config import load_config
from utils.logger import get_logger

sns.set_theme(style='whitegrid')
cfg = load_config()
log = get_logger('training')

## 1. Load Master Features

In [ ]:
master = pd.read_csv('../data/processed/feature_cache/master_features.csv')
print(f'Loaded: {master.shape} | Default rate: {master["TARGET"].mean():.2%}')

# Prepare feature matrix
target_col = 'TARGET'
drop_cols = [target_col, 'SK_ID_CURR']
obj_cols = master.select_dtypes('object').columns.tolist()
high_missing = master.columns[master.isnull().mean() > 0.5].tolist()
X = master.drop(columns=drop_cols + obj_cols + high_missing, errors='ignore')
X = X.fillna(X.median(numeric_only=True))
y = master[target_col]
print(f'Feature matrix: {X.shape}')

## 2. Time-Aware Split (preserves chronological order)

In [ ]:
n = len(X)
train_end = int(n * 0.60)
val_end   = int(n * 0.80)

X_train, y_train = X.iloc[:train_end],  y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test,  y_test  = X.iloc[val_end:],    y.iloc[val_end:]

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}')
print(f'Default rates — Train: {y_train.mean():.2%}  Val: {y_val.mean():.2%}  Test: {y_test.mean():.2%}')

## 3. Baseline: Logistic Regression

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
lr.fit(X_train_sc, y_train)
lr_proba = lr.predict_proba(X_test_sc)[:, 1]

lr_auc = roc_auc_score(y_test, lr_proba)
lr_gini = 2 * lr_auc - 1
lr_ks = _ks_statistic(y_test.values, lr_proba)
print(f'Logistic Regression — AUC: {lr_auc:.4f} | Gini: {lr_gini:.4f} | KS: {lr_ks:.4f}')

## 4. XGBoost Classifier

In [ ]:
clf = DefaultClassifier(cfg)
clf.fit(X_train, y_train, X_val, y_val, calibrate=True)

xgb_proba = clf.predict_proba(X_test)
xgb_auc  = roc_auc_score(y_test, xgb_proba)
xgb_gini = 2 * xgb_auc - 1
xgb_ks   = _ks_statistic(y_test.values, xgb_proba)
print(f'XGBoost — AUC: {xgb_auc:.4f} | Gini: {xgb_gini:.4f} | KS: {xgb_ks:.4f}')

## 5. ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for label, proba, color in [
    (f'XGBoost (AUC={xgb_auc:.3f}, Gini={xgb_gini:.3f})', xgb_proba, '#e74c3c'),
    (f'Logistic Regression (AUC={lr_auc:.3f})', lr_proba, '#3498db'),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=label, linewidth=2, color=color)

ax.plot([0,1],[0,1],'--', color='grey', alpha=0.6, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/03_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Cost-Benefit Threshold Optimisation → Traffic Light Bands

In [ ]:
# Sweep report on validation set
val_proba = clf.predict_proba(X_val)
sweep = threshold_sweep_report(y_val.values, val_proba,
                                cost_fp=cfg.traffic_light.cost_false_positive,
                                cost_fn=cfg.traffic_light.cost_false_negative)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sweep['threshold'], sweep['profit'], color='#2ecc71', linewidth=2)
axes[0].axvline(sweep.loc[sweep['profit'].idxmax(), 'threshold'],
                color='red', linestyle='--', label='Optimal threshold')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Expected Profit ($)')
axes[0].set_title('Profit vs. Decision Threshold', fontweight='bold')
axes[0].legend()

axes[1].plot(sweep['threshold'], sweep['precision'], label='Precision', color='#3498db')
axes[1].plot(sweep['threshold'], sweep['recall'],    label='Recall',    color='#e74c3c')
axes[1].set_xlabel('Threshold')
axes[1].set_title('Precision & Recall vs. Threshold', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/03_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

# Apply optimal thresholds
thresholds = optimise_thresholds(y_val.values, val_proba,
                                  cost_fp=cfg.traffic_light.cost_false_positive,
                                  cost_fn=cfg.traffic_light.cost_false_negative)
clf.red_threshold = thresholds['red_threshold']
clf.green_threshold = thresholds['green_threshold']
print(f'Red threshold: {clf.red_threshold:.3f} | Green threshold: {clf.green_threshold:.3f}')

## 7. Traffic Light Band Validation

In [ ]:
tl_test = clf.predict_traffic_light(X_test)
tl_test['TARGET'] = y_test.values

band_stats = tl_test.groupby('risk_band').agg(
    n=('TARGET','count'),
    default_rate=('TARGET','mean'),
    avg_risk_score=('risk_score','mean'),
).reset_index()

print('Traffic Light Band Validation:')
print(band_stats.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
colors = {'GREEN':'#2ecc71','YELLOW':'#f39c12','RED':'#e74c3c'}
for _, row in band_stats.iterrows():
    ax.bar(row['risk_band'], row['default_rate'],
           color=colors.get(row['risk_band'],'grey'), edgecolor='white', linewidth=1.5)
ax.set_ylabel('Default Rate')
ax.set_title('Actual Default Rate by Traffic Light Band', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
plt.tight_layout()
plt.savefig('../outputs/figures/03_traffic_light_validation.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Model Artefacts

In [ ]:
import os
os.makedirs('../outputs/models', exist_ok=True)
clf.feature_columns = list(X_train.columns)
clf.save('../outputs/models')

# Save evaluation report
report = evaluate_model(clf, X_test, y_test, cfg)
os.makedirs('../outputs/reports', exist_ok=True)
report.to_csv('../outputs/reports/evaluation_report.csv', index=False)
print('All artefacts saved.')
report